# Segment Specimens

Finds every beetle in every photo of a dataset and cuts each one out along its outline, clip-art style.

1. **Grounding DINO** looks for "beetle" in each photo and draws a box around every one it finds.
   - **No beetle found** (e.g. a photo of just tape and a label): the photo is set aside in a `no_beetle/` folder for you to review instead of being used.
   - **Several beetles found**: each beetle becomes its own image.
2. **SAM** (Segment Anything) traces the exact outline of the beetle inside each box.
3. Each beetle is cropped to its outline and saved twice:
   - a **training-ready JPG** on a plain background, in the same folder layout as the source, so `Train_Model.ipynb` can use it directly
   - a **transparent PNG** cutout.

**Output**, for a source folder `Data/My_Dataset/`:

```
Data/My_Dataset_segmented/                 training-ready JPGs, same layout as the source
    training/Carabidae/<photo>.jpg           (one beetle in the photo)
    training/Carabidae/<photo>_beetle01.jpg  (several beetles: numbered top-to-bottom)
    ...
Data/My_Dataset_segmented_extras/
    cutouts/...                              transparent PNGs, same layout
    no_beetle/...                            copies of photos where no beetle was found
    segmentation_log.csv                     one row per photo: what was found and saved
```

Your original photos are never modified.

**Tip:** run `Split_Dataset.ipynb` *first*, then segment the split dataset. That way every beetle from a multi-beetle photo lands in the same split (training or testing), so the model is never tested on a beetle it was trained on.

**Kernel:** select the **specimen** environment (top-right of the notebook). Besides TensorFlow for the other notebooks, it has PyTorch and Hugging Face `transformers`, which Grounding DINO and SAM need. The first run downloads the two models (about 1 GB); after that they load from your disk.

**Speed:** on a CPU, expect roughly 10–25 seconds per photo, so about 3–7 hours per 1,000 photos. Step 4 can be stopped and resumed, so it doesn't have to run in one sitting.

**How to use it:** edit Step 1, run Steps 2 and 3 to preview a few photos, adjust the thresholds if needed, then run Step 4 to process everything.

## Step 1: Settings

- **`SOURCE_DIR`**: the folder to process, e.g. `"My_Dataset"` for `Data/My_Dataset/`. Every image in it and in all its subfolders is processed. Leave it as `None` to pick the folder with a pop-up.
- **`OUTPUT_NAME`**: name for the output folder in `Data/`. `None` uses the source folder's name plus `_segmented`.
- **`PROMPT`**: what Grounding DINO looks for. End it with a full stop.
- **`BOX_THRESHOLD`**: how confident Grounding DINO must be to count something as a beetle. **Lower** it if beetles are being missed; **raise** it if tape, labels or scale bars are being picked up as beetles.
- **`TEXT_THRESHOLD`**: how closely a box must match the words in `PROMPT`. Rarely needs changing.
- **`BACKGROUND_COLOR`**: the colour behind the beetle in the training-ready JPGs, as (red, green, blue). `(255, 255, 255)` is white.
- **`PADDING`**: pixels of background kept around each beetle's outline.
- **`MAKE_SQUARE`**: `Train_Model.ipynb` resizes every image to 224×224, which *stretches* anything that isn't square, so a long, thin beetle would be squashed. With `True`, each training JPG is padded to a square with `BACKGROUND_COLOR` (beetle centred), so that resize only scales the beetle. The images keep their original resolution; `Train_Model.ipynb` does the resizing. `False` keeps the tight crop. The transparent PNG cutouts are never padded.
- **`PREVIEW_IMAGES`**: how many random photos Step 3 previews.

The **advanced** settings rarely need changing; each one has a comment explaining it.

In [ ]:
# ======================= EDIT THESE =======================

SOURCE_DIR = None             # e.g. "My_Dataset"; None -> pick with a pop-up
OUTPUT_NAME = None            # None -> "<source name>_segmented"

PROMPT = "beetle."
BOX_THRESHOLD = 0.35          # lower -> finds more beetles (and more false ones)
TEXT_THRESHOLD = 0.25

BACKGROUND_COLOR = (255, 255, 255)
PADDING = 10                  # pixels
MAKE_SQUARE = True            # pad training JPGs to a square so Train_Model's resize doesn't stretch them

PREVIEW_IMAGES = 6

# ----------------------- advanced -------------------------

MIN_BOX_FRACTION = 0.001      # ignore boxes smaller than this fraction of the photo (dust, specks)
DUPLICATE_IOU = 0.5           # two boxes overlapping more than this are the same beetle
DUPLICATE_CONTAINED = 0.8     # ...as are two boxes where one is 80% inside the other
BOX_EXPAND = 0.05             # grow each box by 5% before tracing, so antenna and leg tips aren't cut off
HOLE_MATCH = 0.5              # fill a hole in the outline if it's at least half as different from the background as the beetle is
TINY_HOLE_FRACTION = 0.005    # always fill holes smaller than 0.5% of the beetle (glare spots)
MIN_PIECE_FRACTION = 0.01     # drop stray bits of outline smaller than 1% of the beetle
RANDOM_SEED = 42              # which photos Step 3 previews

# ==========================================================

## Step 2: Choose the folder and load the models

Finds the folder to process (opening the picker if `SOURCE_DIR` is `None`) and loads Grounding DINO and SAM.

What you'll see, in order:
1. **"Loading libraries..."**: importing PyTorch and transformers takes about 30 seconds in a fresh kernel. The cell is working even though nothing else appears yet.
2. **"A folder picker has opened..."**: choose the folder. The picker sometimes opens behind your other windows, so check your taskbar or press Alt+Tab if you don't see it. **The Interrupt button won't work while the picker is open.**
3. **"Loading models..."**: takes a minute the first time, while the models download, and a few seconds after that.

In [ ]:
print("Loading libraries (about 30 seconds in a fresh kernel)...", flush=True)

import os
import csv
import time
import random
import shutil
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps
from scipy import ndimage
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor, SamModel, SamProcessor

print("Libraries loaded.", flush=True)

DINO_MODEL = "IDEA-Research/grounding-dino-tiny"
SAM_MODEL = "facebook/sam-vit-base"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}


def find_repo_root(marker='Models'):
    path = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return os.path.abspath(os.getcwd())
        path = parent
    return os.path.abspath(os.getcwd())


def pick_folder(start_dir):
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    # Bring the picker in front of the editor/browser. Without parent=root and
    # update(), Windows often opens it hidden behind them.
    root.attributes("-topmost", True)
    root.update()
    folder = filedialog.askdirectory(
        parent=root,
        title="Choose the folder of photos to segment",
        initialdir=start_dir)
    root.destroy()
    return folder


def overlaps(a, b):
    """True if folders a and b are the same folder or one is inside the other."""
    a, b = os.path.normcase(a), os.path.normcase(b)
    try:
        return os.path.commonpath([a, b]) in (a, b)
    except ValueError:  # different drives
        return False


REPO_ROOT = find_repo_root()
DATA_DIR = os.path.join(REPO_ROOT, "Data")

# --- Source folder ---
if SOURCE_DIR is None:
    print("A folder picker has opened. Check your taskbar if you don't see it.", flush=True)
    SOURCE_DIR = pick_folder(DATA_DIR)
    if not SOURCE_DIR:
        raise RuntimeError("No folder was chosen. Run this cell again and pick a folder.")
if not os.path.isabs(SOURCE_DIR):
    SOURCE_DIR = os.path.join(DATA_DIR, SOURCE_DIR)
SOURCE_DIR = os.path.abspath(SOURCE_DIR)
if not os.path.isdir(SOURCE_DIR):
    raise FileNotFoundError(f"This folder doesn't exist: {SOURCE_DIR}")

# --- Output folders ---
if OUTPUT_NAME is None:
    OUTPUT_NAME = os.path.basename(SOURCE_DIR) + "_segmented"
OUTPUT_DIR = os.path.join(DATA_DIR, OUTPUT_NAME)
EXTRAS_DIR = OUTPUT_DIR + "_extras"
for folder in (OUTPUT_DIR, EXTRAS_DIR):
    if overlaps(SOURCE_DIR, folder):
        raise ValueError(
            f"The source folder and {folder} can't be the same folder or inside one another. "
            "Choose a different source folder or OUTPUT_NAME.")

# --- Find every image ---
IMAGE_PATHS = []
for dirpath, dirnames, filenames in os.walk(SOURCE_DIR):
    dirnames[:] = sorted(d for d in dirnames if not d.startswith("."))
    for f in sorted(filenames):
        if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS:
            IMAGE_PATHS.append(os.path.join(dirpath, f))
if not IMAGE_PATHS:
    raise RuntimeError(f"No images found in {SOURCE_DIR} or its subfolders.")

print(f"Source: {SOURCE_DIR}  ({len(IMAGE_PATHS)} images)")
print(f"Output: {OUTPUT_DIR}")
print(f"Extras: {EXTRAS_DIR}")

# --- Load the models ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nLoading models on {DEVICE.upper()} (the first run downloads about 1 GB)...", flush=True)
dino_processor = AutoProcessor.from_pretrained(DINO_MODEL)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL).to(DEVICE).eval()
sam_processor = SamProcessor.from_pretrained(SAM_MODEL)
sam_model = SamModel.from_pretrained(SAM_MODEL).to(DEVICE).eval()
print("Models loaded.")

### The pipeline

This cell only defines the steps; nothing runs yet. You don't need to change anything here.

- **`detect_beetles`**: runs Grounding DINO, then removes boxes that are too small, boxes that wrap a whole group of beetles (Grounding DINO sometimes boxes the group as well as each beetle), and duplicate boxes around the same beetle.
- **`segment_beetles`**: gives each box to SAM, which suggests 3 possible outlines. SAM's own favourite is often wrong (sometimes it's the background around the beetle), so instead the outline whose inside differs most in colour from the background is chosen. The background colour is measured along the edge of the box. Holes inside the outline are filled if they look like beetle (not background, like the gap inside a looped antenna), and stray specks are dropped.
- **`make_cutouts`**: crops the photo to each outline and builds the transparent PNG and the solid-background JPG. With `MAKE_SQUARE`, the JPG is then padded to a square, so the CNN sees the beetle's true shape.

In [ ]:
def box_area(b):
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def box_iou(a, b):
    inter = box_area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))
    union = box_area(a) + box_area(b) - inter
    return inter / union if union else 0.0


def box_containment(a, b):
    """Fraction of the smaller box that lies inside the other one."""
    inter = box_area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))
    smaller = min(box_area(a), box_area(b))
    return inter / smaller if smaller else 0.0


def contains_center(outer, inner):
    cx, cy = (inner[0] + inner[2]) / 2, (inner[1] + inner[3]) / 2
    return outer[0] <= cx <= outer[2] and outer[1] <= cy <= outer[3]


@torch.no_grad()
def detect_beetles(image):
    """Returns [(score, [x0, y0, x1, y1]), ...], one per beetle, top-to-bottom."""
    # Grounding DINO normally enlarges photos to 800 px on the short side first. That
    # triples the time for small lab photos without finding beetles any better.
    size = {"shortest_edge": min(800, image.width, image.height), "longest_edge": 1333}
    inputs = dino_processor(images=image, text=PROMPT, size=size, return_tensors="pt").to(DEVICE)
    outputs = dino_model(**inputs)
    result = dino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
        target_sizes=[image.size[::-1]])[0]
    detections = sorted(zip(result["scores"].tolist(), result["boxes"].tolist()), reverse=True)

    min_area = MIN_BOX_FRACTION * image.width * image.height
    detections = [(s, b) for s, b in detections if box_area(b) >= min_area]

    # A box that holds the centres of 2+ other boxes is around the group, not one beetle
    detections = [(s, b) for s, b in detections
                  if sum(contains_center(b, other) for _, other in detections if other is not b) < 2]

    kept = []
    for s, b in detections:  # most confident first, so duplicates lose to the better box
        if all(box_iou(b, k) < DUPLICATE_IOU and box_containment(b, k) < DUPLICATE_CONTAINED for _, k in kept):
            kept.append((s, b))
    kept.sort(key=lambda d: (d[1][1], d[1][0]))
    return kept


def expand_box(b, width, height):
    dx, dy = (b[2] - b[0]) * BOX_EXPAND, (b[3] - b[1]) * BOX_EXPAND
    return [max(0.0, b[0] - dx), max(0.0, b[1] - dy), min(width, b[2] + dx), min(height, b[3] + dy)]


def box_slice(box, height, width):
    x0, y0 = max(0, int(box[0])), max(0, int(box[1]))
    x1, y1 = min(width, int(np.ceil(box[2]))), min(height, int(np.ceil(box[3])))
    return np.s_[y0:y1, x0:x1]


def background_distance(pixels, box):
    """How different each pixel's colour is from the background. The background colour is
    taken from the box's border, which (after expand_box) is almost never on the beetle."""
    inside = pixels[box_slice(box, *pixels.shape[:2])]
    border = np.concatenate([inside[0], inside[-1], inside[:, 0], inside[:, -1]]).astype(np.float32)
    return np.linalg.norm(pixels.astype(np.float32) - np.median(border, axis=0), axis=2)


def choose_mask(candidates, distance, box):
    """Pick the SAM candidate that stands out most from the background inside the box.
    SAM's own confidence often prefers a patch of background, so it isn't used here."""
    region = box_slice(box, *distance.shape)
    best, best_score = None, 0.0  # a candidate must at least differ more inside than outside
    for mask in candidates:
        m = mask[region]
        if m.all() or not m.any():
            continue
        score = distance[region][m].mean() - distance[region][~m].mean()
        if score > best_score:
            best, best_score = mask, score
    return best


def clean_mask(mask, distance):
    # Fill holes inside the outline, but only ones that look like beetle (e.g. a body SAM
    # traced around) or are tiny (glare), not background (e.g. the gap inside a looped antenna)
    filled = ndimage.binary_fill_holes(mask)
    holes, n = ndimage.label(filled & ~mask)
    if n:
        ids = np.arange(1, n + 1)
        hole_sizes = ndimage.sum(np.ones_like(mask), holes, ids)
        hole_colour = ndimage.mean(distance, holes, ids)
        fill = (hole_colour >= HOLE_MATCH * distance[mask].mean()) | (hole_sizes <= TINY_HOLE_FRACTION * mask.sum())
        mask = mask | np.isin(holes, ids[fill])

    labels, n = ndimage.label(mask)
    if n > 1:
        sizes = ndimage.sum(mask, labels, range(1, n + 1))
        keep = np.flatnonzero(sizes >= MIN_PIECE_FRACTION * sizes.max()) + 1
        mask = np.isin(labels, keep)
    return mask


@torch.no_grad()
def segment_beetles(image, boxes):
    """Returns one True/False mask (height x width) per box, or None where no outline stood out."""
    boxes = [expand_box(b, image.width, image.height) for b in boxes]
    inputs = sam_processor(images=image, input_boxes=[boxes], return_tensors="pt").to(DEVICE)
    outputs = sam_model(**inputs, multimask_output=True)  # 3 candidate outlines per box
    candidates = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(), inputs["reshaped_input_sizes"].cpu())[0]
    pixels = np.asarray(image)
    masks = []
    for box, box_candidates in zip(boxes, candidates):
        distance = background_distance(pixels, box)
        mask = choose_mask([c.numpy() for c in box_candidates], distance, box)
        masks.append(None if mask is None else clean_mask(mask, distance))
    return masks


def pad_to_square(image):
    """Centre the image on a square of BACKGROUND_COLOR, at its original resolution.
    Keeps the beetle's proportions, so Train_Model's resize to 224x224 doesn't stretch it."""
    side = max(image.size)
    square = Image.new("RGB", (side, side), BACKGROUND_COLOR)
    square.paste(image, ((side - image.width) // 2, (side - image.height) // 2))
    return square


def make_cutouts(image, mask):
    """Returns (transparent PNG, training image), cropped to the mask. None if the mask is empty."""
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    x0, y0 = max(0, xs.min() - PADDING), max(0, ys.min() - PADDING)
    x1, y1 = min(image.width, xs.max() + 1 + PADDING), min(image.height, ys.max() + 1 + PADDING)
    rgb = np.asarray(image)[y0:y1, x0:x1]
    alpha = mask[y0:y1, x0:x1].astype(np.uint8) * 255
    png = Image.fromarray(np.dstack([rgb, alpha]), "RGBA")
    solid = Image.new("RGB", png.size, BACKGROUND_COLOR)
    solid.paste(png, mask=png.getchannel("A"))
    if MAKE_SQUARE:
        solid = pad_to_square(solid)
    return png, solid


def process_image(path):
    """Returns (image, [(score, box, png, solid), ...]) with one entry per beetle found."""
    image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    detections = detect_beetles(image)
    if not detections:
        return image, []
    masks = segment_beetles(image, [b for _, b in detections])
    beetles = []
    for (score, box), mask in zip(detections, masks):
        if mask is None:
            continue
        cutouts = make_cutouts(image, mask)
        if cutouts is not None:
            beetles.append((score, box, *cutouts))
    return image, beetles

## Step 3: Preview

Runs the pipeline on a few random photos and shows, for each one, the photo with the boxes Grounding DINO found (and how confident it was) next to each beetle's **training image**: exactly what Step 4 saves for `Train_Model.ipynb`, padded to a square when `MAKE_SQUARE` is on. **Nothing is saved.**

Check that:
- every beetle has a box, and nothing that isn't a beetle does. If not, adjust `BOX_THRESHOLD` in Step 1.
- the outlines keep the legs and antennae. If tips are cut off, raise `BOX_EXPAND`.
- the beetles keep their true shape (long beetles stay long) inside the square.

After changing a setting, re-run Step 1 and this cell; you don't need to re-run Step 2. Re-run just this cell to see different photos, or set `RANDOM_SEED` to another number.

In [ ]:
MAX_CUTOUTS_SHOWN = 6

sample = random.Random(RANDOM_SEED).sample(IMAGE_PATHS, min(PREVIEW_IMAGES, len(IMAGE_PATHS)))
RANDOM_SEED += 1  # a different sample next time this cell runs

for path in sample:
    start = time.time()
    image, beetles = process_image(path)
    seconds = time.time() - start

    shown = beetles[:MAX_CUTOUTS_SHOWN]
    fig, axes = plt.subplots(1, 1 + max(len(shown), 1), figsize=(4 * (1 + max(len(shown), 1)), 4),
                             gridspec_kw={"width_ratios": [2] + [1] * max(len(shown), 1)}, squeeze=False)
    axes = axes[0]
    axes[0].imshow(image)
    for n, (score, box, _, _) in enumerate(beetles, 1):
        axes[0].add_patch(Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                    fill=False, edgecolor="red", linewidth=1.5))
        axes[0].text(box[0], box[1] - 3, f"{n}: {score:.2f}", color="red", fontsize=8,
                     bbox={"facecolor": "white", "alpha": 0.7, "pad": 1, "edgecolor": "none"})
    found = f"{len(beetles)} beetle(s)" if beetles else "NO BEETLE: would go to no_beetle/"
    axes[0].set_title(f"{os.path.relpath(path, SOURCE_DIR)}\n{found}  ({seconds:.1f} s)", fontsize=8)

    for n, ax in enumerate(axes[1:], 1):
        if n <= len(shown):
            training_image = shown[n - 1][3]  # exactly what gets saved for Train_Model
            ax.imshow(training_image)
            ax.set_title(f"beetle {n}  ({training_image.width}x{training_image.height})", fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor("#9a9a9a")  # frame, so the white padding's edge is visible
        else:
            ax.axis("off")
    axes[0].axis("off")
    if len(beetles) > MAX_CUTOUTS_SHOWN:
        fig.suptitle(f"(showing the first {MAX_CUTOUTS_SHOWN} of {len(beetles)} cutouts)", fontsize=8)
    plt.tight_layout()
    plt.show()

## Step 4: Process every photo

Runs the pipeline on every photo and saves the results (see the folder layout at the top). Progress and an estimated time remaining are printed every 10 photos.

**You can stop and resume.** Every finished photo is recorded in `segmentation_log.csv` straight away, so if you interrupt this cell (or close VS Code), running it again carries on where it stopped. To redo everything from scratch, for example after changing the thresholds, delete the two output folders first or set a new `OUTPUT_NAME`.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXTRAS_DIR, exist_ok=True)
log_path = os.path.join(EXTRAS_DIR, "segmentation_log.csv")

new_log = not os.path.exists(log_path) or os.path.getsize(log_path) == 0
done = set()
if not new_log:
    with open(log_path, newline="", encoding="utf-8") as fh:
        # Photos that errored last time are tried again
        done = {row["source"] for row in csv.DictReader(fh) if not row["status"].startswith("error")}

todo = [p for p in IMAGE_PATHS if os.path.relpath(p, SOURCE_DIR) not in done]
print(f"{len(IMAGE_PATHS)} photos: {len(done)} already done, {len(todo)} to go.\n")

counts = Counter()
start = time.time()
with open(log_path, "a", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    if new_log:
        writer.writerow(["source", "status", "beetles", "scores", "outputs"])

    for i, path in enumerate(todo, 1):
        rel_path = os.path.relpath(path, SOURCE_DIR)
        rel_dir, filename = os.path.split(rel_path)
        stem = os.path.splitext(filename)[0]

        try:
            image, beetles = process_image(path)
        except Exception as e:
            writer.writerow([rel_path, f"error: {e}", 0, "", ""])
            counts["error"] += 1
            print(f"ERROR on {rel_path}: {e}")
            continue

        if not beetles:
            dest_dir = os.path.join(EXTRAS_DIR, "no_beetle", rel_dir)
            os.makedirs(dest_dir, exist_ok=True)
            shutil.copy2(path, os.path.join(dest_dir, filename))
            writer.writerow([rel_path, "no_beetle", 0, "", ""])
            counts["no_beetle"] += 1
        else:
            jpg_dir = os.path.join(OUTPUT_DIR, rel_dir)
            png_dir = os.path.join(EXTRAS_DIR, "cutouts", rel_dir)
            os.makedirs(jpg_dir, exist_ok=True)
            os.makedirs(png_dir, exist_ok=True)
            names = []
            for n, (score, box, png, solid) in enumerate(beetles, 1):
                name = f"{stem}_beetle{n:02d}" if len(beetles) > 1 else stem
                solid.save(os.path.join(jpg_dir, name + ".jpg"), quality=95)
                png.save(os.path.join(png_dir, name + ".png"))
                names.append(name)
            writer.writerow([rel_path, "ok", len(beetles),
                             ";".join(f"{s:.2f}" for s, *_ in beetles), ";".join(names)])
            counts["ok"] += 1
            counts["beetles"] += len(beetles)
        fh.flush()

        if i % 10 == 0 or i == len(todo):
            elapsed = time.time() - start
            remaining = elapsed / i * (len(todo) - i)
            print(f"{i}/{len(todo)} photos  |  {elapsed / 60:.1f} min so far, about {remaining / 60:.1f} min left")

print(f"\nDone. {counts['ok']} photos gave {counts['beetles']} beetle cutouts.")
if counts["no_beetle"]:
    print(f"{counts['no_beetle']} photos had no beetle; copies are in {os.path.join(EXTRAS_DIR, 'no_beetle')} for review.")
if counts["error"]:
    print(f"{counts['error']} photos couldn't be processed; see the 'status' column of {log_path}.")
print(f"\nTo train on the cutouts, set this line in Train_Model.ipynb:")
print(f'    dataset_dir = os.path.join(REPO_ROOT, "Data", "{OUTPUT_NAME}")')